# Import 

In [ ]:
# Xây dựng mô hình time-series forecasting cho bike-sharing dataset:

# Import libraries:
import numpy as np 
import pandas as pd
from ydata_profiling import ProfileReport
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
!pip install prophet
from prophet import Prophet

# Import dataset:
bike_df = pd.read_csv(f'/Users/ngocta/Desktop/CoderSchool-AI/Week9/bike-sharing-dataset.csv')
bike_df['date_time'] = pd.to_datetime(bike_df['date_time'], format = '%Y-%m-%d %H:%M:%S')
target = 'users'
print(bike_df.head(10))
print(bike_df.dtypes)
print(bike_df.info())

# Checking data statistics:
# bike_profile = ProfileReport(bike_df, title = "Bike Report", explorative = True)
# bike_profile.to_file("Bike Report.html")

# Visualize dataset:
fig, ax = plt.subplots()
ax.plot(bike_df['date_time'], bike_df[target])
ax.set_xlabel('Time')
ax.set_ylabel('Users')
ax.set_title('User growth')
plt.show()

# Break users into 5 windows:
window_size = 5
i = 1
while i < window_size:
    bike_df["users_{}".format(i)] = bike_df[target].shift(-i)
    i += 1
bike_df[target] = bike_df[target].shift(-i)

# Drop rows with empty cells:
bike_df.dropna(inplace = True)

# Check correlation of numericcal columns & visualize with heatmap:
corr = bike_df.drop(['date_time', 'weather'], axis = 1).corr()
plt.figure(figsize = (12, 10))
sns.heatmap(corr, annot = True, cmap = 'coolwarm', fmt = ".2f", linewidths = .5)
plt.title('Correlation Matrix of Bike Sharing Features')
plt.show()

# Split into train & test sets:
x = bike_df.drop([target, 'date_time'], axis = 1)
y = bike_df[target]
num_samples = len(x)
train_ratio = 0.8

x_train = x[:int(num_samples * train_ratio)] 
y_train = y[:int(num_samples * train_ratio)] 
x_test = x[int(num_samples * train_ratio):] 
y_test = y[int(num_samples * train_ratio):]

# Preprocessing columns:
categorical_cols = ['weather']
excluded_cols = ['date_time', 'weather', target]
numerical_cols = [col for col in bike_df.columns if col not in excluded_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(handle_unknown = 'ignore'), categorical_cols),
        ('numerical', StandardScaler(), numerical_cols)
    ])

print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)

In [ ]:
# TRAIN WITH REGRESSION MODELS:
# Define your models
regressors = {
    'Linear Regression': LinearRegression(),
    'Support Vector Regressor': SVR(),
    'Decision Tree Regressor': DecisionTreeRegressor(),
    'Random Forest Regressor': RandomForestRegressor(),
    'Gradient Boosting Regressor': GradientBoostingRegressor()
}

for name, model in regressors.items():
    print(f"\n{name}:")
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor), 
        ('regressor', model)
    ])
    # Train:
    pipe.fit(x_train, y_train)
    
    # Predict:
    y_pred = pipe.predict(x_test)

    # Calculate performance:
    mse = mean_squared_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred, squared = False)
    mae = mean_absolute_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)
    
    # Report results with 4 decimal points:
    print(f"Root Mean Squared Error: {rmse:.4f}")
    print(f"Mean Squared Error: {mean_squared_error(y_test, y_pred):.4f}")
    print(f"Mean Absolute Error: {mean_absolute_error(y_test, y_pred):.4f}")
    print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")

# Conclusion: for non-timeseries models, the best performing one is Random Forest Regressor because 
# it had the lowest Error scores (RMSE, MSE, MAE) and highest R2 Score (closest to 1)

In [ ]:
# TRAIN WITH TIME SERIES MODEL - FACEBOOK PROPHET:
prophet_df = bike_df[['date_time', 'users']].rename(columns={'date_time':'ds','users':'y'})

# Split into train/test
train_size = int(len(prophet_df) * 0.8)
train_df = prophet_df.iloc[:train_size]
test_df  = prophet_df.iloc[train_size:]

# Train with Facebook Prophet:
model_prophet = Prophet()
model_prophet.fit(train_df)
future = model_prophet.make_future_dataframe(periods = len(test_df), freq = 'h')

# Predict
forecast = model_prophet.predict(future)

# Make hourly future frame for test period:
pred_df = forecast[['ds', 'yhat']].set_index('ds')
yhat_test = pred_df.loc[test_df['ds'], 'yhat'].values

# Calculate RMSE:
y_true = test_df['y'].values
rmse = np.sqrt(mean_squared_error(y_true, yhat_test))
print(f"Prophet RMSE: {rmse:.4f}")

# Plot the forecast vs. ground truth
fig, ax = plt.subplots(figsize = (20,5))
sns.lineplot(x = test_df['ds'], y = yhat_test, ax = ax, label = 'Forecast')
sns.lineplot(x = test_df['ds'], y = y_true, ax = ax, color = 'orange', label = 'Ground truth')
ax.set_title(f'Prophet RMSE: {rmse:.4f}', fontsize = 14)
ax.set_xlabel('Hour',  fontsize = 14)
ax.set_ylabel('Users', fontsize = 14)
ax.legend()
plt.show()

# Conclusion: The best performing one is still Random Forest Regressor for its RMSE is lower than Facebook Prophet